## Setup & Configuration

In [0]:
# --- SETUP: Define Paths using Unity Catalog Volume ---
from pyspark.sql.functions import col, current_timestamp, to_date

# Pointing the entire pipeline to the Volume for enterprise-grade processing
BASE_PATH = "/Volumes/workspace/default/retailstream_vol"

# Data paths (Raw files you uploaded)
data_path       = BASE_PATH + '/data'
batch_initial_p = data_path + '/batch_initial'
batch_incr_p    = data_path + '/batch_incremental'
late_arriving_p = data_path + '/late_arriving'
autoloader_p    = data_path + '/autoloader_landing'

# Dimension files
stores_file    = data_path + '/stores.csv'
products_file  = data_path + '/products.csv'
customers_file = data_path + '/customers.csv'

# Output paths (Delta tables & Checkpoints)
bronze_orders_path = BASE_PATH + '/delta/bronze/bronze_orders'
bronze_txn_path    = BASE_PATH + '/delta/bronze/bronze_transactions'
silver_orders_path = BASE_PATH + '/delta/silver/silver_orders'
gold_monthly_path  = BASE_PATH + '/delta/gold/gold_monthly_sales'
gold_payment_path  = BASE_PATH + '/delta/gold/gold_payment_summary'
uc_checkpoint_path = BASE_PATH + '/checkpoints/bronze_txn'

print(f"Pipeline successfully configured to run using Unity Catalog Volume: {BASE_PATH}")

Pipeline successfully configured to run using Unity Catalog Volume: /Volumes/workspace/default/retailstream_vol


## Task 1 — Initial Batch Load (Bronze Layer)
This task reads the initial January orders using a wildcard path (`/*.csv`). We cast the `order_date` to a proper date format and append tracking metadata (`ingested_at` and the Unity Catalog `_metadata.file_path`). The data is written to the Bronze layer in `overwrite` mode.

In [0]:
# ---- TASK 1: INITIAL BATCH LOAD (Bronze) ----
jan_df = (
    spark.read.csv(batch_initial_p + '/*.csv', header=True, inferSchema=True)
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))
    .withColumn('ingested_at', current_timestamp())
    .withColumn('source_file', col('_metadata.file_path')) 
)

# Overwrite for the initial load
jan_df.write.format('delta').mode('overwrite').save(bronze_orders_path)

# Verify count = 20
bronze_orders = spark.read.format('delta').load(bronze_orders_path)
print(f'Bronze orders count after Task 1: {bronze_orders.count()}')
bronze_orders.show(5, truncate=False)

Bronze orders count after Task 1: 20
+--------+-----------+----------+--------+----------+----------+--------+---------+--------------------------+--------------------------------------------------------------------------------------+
|order_id|customer_id|product_id|quantity|unit_price|order_date|store_id|status   |ingested_at               |source_file                                                                           |
+--------+-----------+----------+--------+----------+----------+--------+---------+--------------------------+--------------------------------------------------------------------------------------+
|ORD001  |C026       |P008      |1       |28000     |2024-01-11|S02     |DELIVERED|2026-08-14 14:48:58.015133|dbfs:/Volumes/workspace/default/retailstream_vol/data/batch_initial/orders_2024_01.csv|
|ORD002  |C021       |P004      |4       |18000     |2024-01-12|S03     |DELIVERED|2026-08-14 14:48:58.015133|dbfs:/Volumes/workspace/default/retailstream_vol/data/batch_i

## Task 2 — Incremental Batch Load (Append Only)
This task loads the February orders and prevents duplication of records (like `ORD003` and `ORD013`) using a `left_anti` join against the existing Bronze table based on `order_id`. Only genuinely new records are appended to the table.

In [0]:
# ---- TASK 2: INCREMENTAL BATCH LOAD (No duplicates) ----
feb_df = (
    spark.read.csv(batch_incr_p + '/*.csv', header=True, inferSchema=True)
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))
    .withColumn('ingested_at', current_timestamp())
    .withColumn('source_file', col('_metadata.file_path'))
)

# Read existing bronze table to check for duplicates
existing = spark.read.format('delta').load(bronze_orders_path)

# Anti-join to keep only new records
new_records = feb_df.join(existing.select('order_id'), 'order_id', 'left_anti')

# Append safely
new_records.write.format('delta').mode('append').save(bronze_orders_path)

# Verify total count = 40
print(f"Bronze orders count after Task 2: {spark.read.format('delta').load(bronze_orders_path).count()}")

Bronze orders count after Task 2: 40


## Task 3 — Auto Loader Streaming Ingestion
Using Databricks `cloudFiles`, this stream continuously ingests transaction CSVs into the Bronze layer as they arrive. An explicit schema is enforced, timestamps are cast appropriately, and state is preserved securely in the Unity Catalog Volume checkpoint directory. It stops automatically via `availableNow=True`.

In [0]:
# ---- TASK 3: AUTO LOADER STREAMING (Bronze transactions) ----
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp

# Clean up any previous state to prevent CF_BUCKET_MISMATCH
dbutils.fs.rm(uc_checkpoint_path, recurse=True)

txn_schema = StructType([
    StructField('txn_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('payment_method', StringType(), True),
    StructField('amount', DoubleType(), True),
    StructField('txn_timestamp', StringType(), True),
    StructField('currency', StringType(), True),
    StructField('gateway_status', StringType(), True)
])

txn_stream = (
    spark.readStream.format('cloudFiles')
    .option('cloudFiles.format', 'csv')
    .option('header', 'true')
    .schema(txn_schema)
    .load(autoloader_p) 
    .withColumn('txn_timestamp', to_timestamp(col('txn_timestamp'), 'yyyy-MM-dd HH:mm:ss'))
)

query = (
    txn_stream.writeStream
    .format('delta')
    .option('checkpointLocation', uc_checkpoint_path) 
    .outputMode('append')
    .trigger(availableNow=True)
    .start(bronze_txn_path) 
)
query.awaitTermination()

# Verify count = 15
bronze_txn = spark.read.format('delta').load(bronze_txn_path)
print(f'Bronze transactions count after Task 3: {bronze_txn.count()}')
bronze_txn.show(5, truncate=False)

Bronze transactions count after Task 3: 45
+------+--------+--------------+--------+-------------------+--------+--------------+
|txn_id|order_id|payment_method|amount  |txn_timestamp      |currency|gateway_status|
+------+--------+--------------+--------+-------------------+--------+--------------+
|TXN011|ORD035  |DEBIT_CARD    |19539.22|2024-03-01 03:00:00|INR     |SUCCESS       |
|TXN012|ORD028  |CREDIT_CARD   |37996.39|2024-03-02 11:00:00|INR     |SUCCESS       |
|TXN013|ORD002  |CREDIT_CARD   |38652.56|2024-03-01 22:00:00|INR     |SUCCESS       |
|TXN014|ORD_L02 |CREDIT_CARD   |30838.17|2024-03-01 04:00:00|INR     |SUCCESS       |
|TXN015|ORD022  |NET_BANKING   |11047.2 |2024-03-02 05:00:00|INR     |FAILED        |
+------+--------+--------------+--------+-------------------+--------+--------------+
only showing top 5 rows


## Task 4 — Late Arriving File Handling (Delta MERGE)
To handle the delayed Store S04 January orders, we perform an idempotent Delta `MERGE` (upsert). We match on `order_id` and use `whenNotMatchedInsertAll()` to safely insert the 5 missing records without duplicating or overwriting the existing baseline data.


In [0]:
# ---- TASK 4: LATE ARRIVING FILE -> DELTA MERGE ----
from delta.tables import DeltaTable

late_df = (
    spark.read.csv(late_arriving_p + '/*.csv', header=True, inferSchema=True)
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd'))
    .withColumn('ingested_at', current_timestamp())
    .withColumn('source_file', col('_metadata.file_path')) 
)

# Merge to prevent duplicating already-loaded data
target = DeltaTable.forPath(spark, bronze_orders_path)
(target.alias('target')
 .merge(late_df.alias('source'), 'target.order_id = source.order_id')
 .whenNotMatchedInsertAll()
 .execute())

# Verify total = 45 and Store S04 = 8 (3 from Feb + 5 Late Jan)
bronze_orders = spark.read.format('delta').load(bronze_orders_path)
print(f'Bronze orders total count after Task 4: {bronze_orders.count()}')
print(f"Records with store_id = S04: {bronze_orders.filter(col('store_id') == 'S04').count()}")

Bronze orders total count after Task 4: 45
Records with store_id = S04: 8


## Task 5 — Silver Layer (Cleaned & Enriched)
This task reads the Bronze orders and joins them with the three static dimension tables (Products, Customers, Stores). It derives core business metrics (`revenue` and `margin`) and drops unnecessary foreign keys before writing out an analytics-ready Silver table.

In [0]:
# ---- TASK 5: SILVER LAYER (Joins + Computed metrics) ----
bronze_orders = spark.read.format('delta').load(bronze_orders_path)
products  = spark.read.csv(products_file, header=True, inferSchema=True)
customers = spark.read.csv(customers_file, header=True, inferSchema=True)
stores    = spark.read.csv(stores_file, header=True, inferSchema=True)

silver_orders = (
    bronze_orders
    .join(products.select('product_id', 'product_name', 'category', 'cost_price'), 'product_id', 'left')
    .join(customers.select('customer_id', 'customer_name', 'city', 'tier'), 'customer_id', 'left')
    .join(stores.select('store_id', 'store_name', 'region'), 'store_id', 'left')
    .withColumn('revenue', col('quantity') * col('unit_price'))
    .withColumn('margin', col('revenue') - (col('quantity') * col('cost_price')))
    .drop('customer_id', 'product_id', 'store_id')
)

silver_orders.write.format('delta').mode('overwrite').save(silver_orders_path)

# Verify count = 45
silver = spark.read.format('delta').load(silver_orders_path)
print(f'Silver orders count: {silver.count()}')
silver.show(5, truncate=False)

Silver orders count: 45
+--------+--------+----------+----------+---------+--------------------------+--------------------------------------------------------------------------------------+------------+-----------+----------+-------------+---------+------+--------------+------+-------+------+
|order_id|quantity|unit_price|order_date|status   |ingested_at               |source_file                                                                           |product_name|category   |cost_price|customer_name|city     |tier  |store_name    |region|revenue|margin|
+--------+--------+----------+----------+---------+--------------------------+--------------------------------------------------------------------------------------+------------+-----------+----------+-------------+---------+------+--------------+------+-------+------+
|ORD001  |1       |28000     |2024-01-11|DELIVERED|2026-08-14 14:48:58.015133|dbfs:/Volumes/workspace/default/retailstream_vol/data/batch_initial/orders_2024_01.csv|S

## Task 6 — Gold Layer (SparkSQL Analytics)
Here we register our Silver orders and Bronze transactions as temporary views to run business-level aggregations using standard SQL. This outputs two distinct Gold tables: a Monthly Sales Summary and a Payment Method Success Summary.

In [0]:
# ---- TASK 6: GOLD LAYER (SparkSQL analytics) ----
spark.read.format('delta').load(silver_orders_path).createOrReplaceTempView('silver_orders')
spark.read.format('delta').load(bronze_txn_path).createOrReplaceTempView('bronze_transactions')

# 6a - Monthly sales summary
gold_monthly = spark.sql('''
    SELECT DATE_FORMAT(order_date, 'yyyy-MM') AS month,
           COUNT(*)               AS total_orders,
           ROUND(SUM(revenue), 2) AS total_revenue,
           ROUND(SUM(margin), 2)  AS total_margin,
           ROUND(AVG(revenue), 2) AS avg_order_value
    FROM silver_orders
    GROUP BY 1
    ORDER BY 1
''')
gold_monthly.write.format('delta').mode('overwrite').save(gold_monthly_path)
print("6a. Monthly Sales Summary:")
gold_monthly.show(truncate=False)

# 6b - Payment method summary
gold_payment = spark.sql('''
    SELECT payment_method,
           COUNT(*) AS total_transactions,
           ROUND(SUM(CASE WHEN gateway_status = 'SUCCESS' THEN 1 ELSE 0 END)
                 * 100.0 / COUNT(*), 2) AS success_rate,
           ROUND(SUM(amount), 2) AS total_amount
    FROM bronze_transactions
    GROUP BY payment_method
    ORDER BY payment_method
''')
gold_payment.write.format('delta').mode('overwrite').save(gold_payment_path)
print("6b. Payment Method Summary:")
gold_payment.show(truncate=False)

6a. Monthly Sales Summary:
+-------+------------+-------------+------------+---------------+
|month  |total_orders|total_revenue|total_margin|avg_order_value|
+-------+------------+-------------+------------+---------------+
|2024-01|25          |731900       |71700       |29276.0        |
|2024-02|20          |959000       |615800      |47950.0        |
+-------+------------+-------------+------------+---------------+

6b. Payment Method Summary:
+--------------+------------------+------------+------------+
|payment_method|total_transactions|success_rate|total_amount|
+--------------+------------------+------------+------------+
|CREDIT_CARD   |9                 |100.00      |322461.36   |
|DEBIT_CARD    |12                |75.00       |337237.26   |
|NET_BANKING   |9                 |66.67       |155572.17   |
|UPI           |15                |60.00       |271311.66   |
+--------------+------------------+------------+------------+

